In [12]:
!git clone https://github.com/ravirajsinh45/Crop_and_weed_detection.git
!ls -la Crop_and_weed_detection


Cloning into 'Crop_and_weed_detection'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 151 (delta 28), reused 10 (delta 10), pack-reused 114 (from 1)
Receiving objects: 100% (151/151), 3.91 MiB | 17.31 MiB/s, done.
Resolving deltas: 100% (63/63), done.
total 44
drwxr-xr-x 5 root root  4096 Nov  1 07:13 .
drwxr-xr-x 6 root root  4096 Nov  1 07:13 ..
drwxr-xr-x 4 root root  4096 Nov  1 07:13 Crop_weed_detection_training
drwxr-xr-x 8 root root  4096 Nov  1 07:13 .git
-rw-r--r-- 1 root root 11357 Nov  1 07:13 LICENSE
drwxr-xr-x 5 root root  4096 Nov  1 07:13 performing_detection
-rw-r--r-- 1 root root  5810 Nov  1 07:13 README.md
-rw-r--r-- 1 root root   261 Nov  1 07:13 requirements.txt


In [13]:
!pip install --quiet opencv-python-headless==4.7.0.72 gradio gdown


In [14]:
%%bash
WEIGHTS_DIR="Crop_and_weed_detection/performing_detection/data/weights"
mkdir -p "$WEIGHTS_DIR"
WEIGHTS_PATH="$WEIGHTS_DIR/crop_weed_detection.weights"

if [ ! -f "$WEIGHTS_PATH" ]; then
  echo "Downloading weights..."
  gdown --id 1-Aam2D-fqnwecbeHwa4rtzxtNjwcDkP6 -O "$WEIGHTS_PATH" || echo "gdown failed — if that happens, download manually and upload to the Colab files panel."
else
  echo "Weights already downloaded."
fi
ls -lh "$WEIGHTS_DIR" || true

total 235M
-rw-r--r-- 1 root root   78 Nov  1 07:13 Add trained weight file here.txt
-rw-r--r-- 1 root root 235M Apr  7  2020 crop_weed_detection.weights


/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1-Aam2D-fqnwecbeHwa4rtzxtNjwcDkP6
From (redirected): https://drive.google.com/uc?id=1-Aam2D-fqnwecbeHwa4rtzxtNjwcDkP6&confirm=t&uuid=79edccfc-fcb3-4c31-ac4b-ebfb20721e39
To: /content/Crop_and_weed_detection/Crop_and_weed_detection/performing_detection/data/weights/crop_weed_detection.weights
100%|██████████| 246M/246M [00:03<00:00, 72.9MB/s]


In [15]:
!pip install numpy==1.26.4

In [5]:
!pip install gdown

In [16]:
# 4) Detection utilities (OpenCV DNN)
import os, glob, cv2, numpy as np
from typing import List, Tuple

REPO = "Crop_and_weed_detection"
WEIGHTS_DIR = os.path.join(REPO, "performing_detection", "data", "weights")
WEIGHTS_PATH = os.path.join(WEIGHTS_DIR, "crop_weed_detection.weights")

# Auto-find .cfg and .names/.txt label file in repo (best-effort)
cfg_files = glob.glob(os.path.join(REPO, "**", "*.cfg"), recursive=True)
names_files = glob.glob(os.path.join(REPO, "**", "*.names"), recursive=True) + glob.glob(os.path.join(REPO, "**", "*.txt"), recursive=True)
weights_files = glob.glob(os.path.join(REPO, "**", "*.weights"), recursive=True)

cfg_path = cfg_files[0] if cfg_files else None
# prefer obvious names files (classes, obj, names)
label_path = None
for f in names_files:
    fn = os.path.basename(f).lower()
    if any(k in fn for k in ("names", "classes", "obj", "label")):
        label_path = f
        break
if not label_path and names_files:
    label_path = names_files[0]

# Ensure weights are downloaded before proceeding
if not os.path.exists(WEIGHTS_PATH):
    print("Weights not found. Please run the previous cell to download them.")
    cfg_path = None # Prevent model loading if weights are missing

# prefer repo weight if present else the downloaded one
if weights_files:
    chosen_weights = weights_files[0]
else:
    chosen_weights = WEIGHTS_PATH if os.path.exists(WEIGHTS_PATH) else None


print("Auto-detected files:")
print(" CFG:", cfg_path)
print(" LABELS:", label_path)
print(" WEIGHTS:", chosen_weights)

def load_labels(path: str) -> List[str]:
    with open(path, "r") as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]
    return lines

def build_net(cfg: str, weights: str):
    net = cv2.dnn.readNetFromDarknet(cfg, weights)
    net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
    # For Colab CPU this is default; if you have OpenCV with OpenCL on your local machine:
    # net.setPreferableTarget(cv2.dnn.DNN_TARGET_OPENCL)
    return net

# Detection function: takes BGR img (numpy), returns annotated image (BGR)
def detect_and_annotate(net, labels: List[str], image: np.ndarray,
                        input_size: int = 416,
                        conf_threshold: float = 0.3,
                        nms_threshold: float = 0.4) -> np.ndarray:
    H, W = image.shape[:2]
    blob = cv2.dnn.blobFromImage(image, 1/255.0, (input_size, input_size), swapRB=True, crop=False)
    net.setInput(blob)
    ln = net.getLayerNames()
    try:
        out_names = [ln[i - 1] for i in net.getUnconnectedOutLayers().flatten()]
    except:
        out_names = [ln[i[0] - 1] for i in net.getUnconnectedOutLayers()]
    layerOutputs = net.forward(out_names)

    boxes, confidences, classIDs = [], [], []
    for output in layerOutputs:
        for detection in output:
            scores = detection[5:]
            classID = int(np.argmax(scores))
            confidence = float(scores[classID])
            if confidence > conf_threshold:
                box = detection[0:4] * np.array([W, H, W, H])
                (centerX, centerY, width, height) = box.astype("int")
                x = int(centerX - (width / 2))
                y = int(centerY - (height / 2))
                boxes.append([x, y, int(width), int(height)])
                confidences.append(confidence)
                classIDs.append(classID)

    idxs = cv2.dnn.NMSBoxes(boxes, confidences, conf_threshold, nms_threshold)
    # annotate
    out = image.copy()
    if len(idxs) > 0:
        for i in idxs.flatten():
            x, y, w, h = boxes[i]
            x1, y1, x2, y2 = max(0,x), max(0,y), min(W, x+w), min(H, y+h)
            label = labels[classIDs[i]] if classIDs[i] < len(labels) else str(classIDs[i])
            color = (0, 255, 0)
            cv2.rectangle(out, (x1,y1), (x2,y2), color, 2)
            text = f"{label}: {confidences[i]:.2f}"
            cv2.putText(out, text, (x1, max(15,y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return out

# If we have model files, build net and labels now (otherwise user must upload)
net = None
labels = []
if cfg_path and chosen_weights and label_path:
    labels = load_labels(label_path)
    net = build_net(cfg_path, chosen_weights)
    print("Model loaded — ready for detection.")
else:
    print("Model files not fully found. If missing, either upload them in the Colab file browser or open the repo notebook and run there.")

Auto-detected files:
 CFG: Crop_and_weed_detection/performing_detection/data/cfg/crop_weed.cfg
 LABELS: Crop_and_weed_detection/performing_detection/data/names/obj.names
 WEIGHTS: Crop_and_weed_detection/performing_detection/data/weights/crop_weed_detection.weights
Model loaded — ready for detection.
Auto-detected files:
 CFG: Crop_and_weed_detection/performing_detection/data/cfg/crop_weed.cfg
 LABELS: Crop_and_weed_detection/performing_detection/data/names/obj.names
 WEIGHTS: Crop_and_weed_detection/performing_detection/data/weights/crop_weed_detection.weights
Model loaded — ready for detection.


In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
!pip install numpy==1.26.4

In [9]:
import gradio as gr
import numpy as np
import cv2
from PIL import Image as PILImage
import io

def gradio_detect(pil_img: PILImage):
    global net, labels
    if pil_img is None:
        return None
    img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    if net is None:
        # If model not loaded, just return original image and instruct user
        msg = "Model not loaded. Upload cfg/names/weights into the Colab files (left panel) or run the repo notebooks."
        # create image with message overlay
        h, w = img.shape[:2]
        cv2.putText(img, msg, (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    out = detect_and_annotate(net, labels, img)
    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)

title = "Crop vs Weed Detection (OpenCV DNN) - Gradio Frontend"
description = "Upload an image or use your camera. The model (YOLO .cfg + .weights + labels) is loaded from the cloned repo if available."

iface = gr.Interface(
    fn=gradio_detect,
    inputs=[gr.Image(type="pil", label="Upload Image or Camera")],
    outputs=[gr.Image(type="numpy", label="Detection Output")],
    title=title,
    description=description,
    allow_flagging="never",
    examples=None,
    live=False,
    analytics_enabled=False
)

# Launch the interface. In Colab use share=True to get a public link hosted by Gradio.
print("Launching Gradio app — this will output a link. Click it to open the front-end.")
iface.launch(share=True, debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Launching Gradio app — this will output a link. Click it to open the front-end.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c773ad03e7c30d2c19.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c773ad03e7c30d2c19.gradio.live


/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Launching Gradio app — this will output a link. Click it to open the front-end.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2a7b729ed3203792b3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [11]:

NGROK_AUTH_TOKEN = "33yAdkQEJ7VcsR5gwDBnlhIsr2E_6eXXVsMu4xpZoBww27Qx7"   # <-- put your token here

# 1) Install dependencies
!apt-get -qq update
!pip install -q streamlit pyngrok opencv-python-headless numpy pillow gdown

# 2) Clone repo if not present
import os, time, glob, shutil
if not os.path.exists("/content/Crop_and_weed_detection"):
    !git clone https://github.com/ravirajsinh45/Crop_and_weed_detection.git
os.chdir("/content/Crop_and_weed_detection")
print("Working directory:", os.getcwd())

# 3) Helper: find files
def find_first(patterns):
    for p in patterns:
        res = glob.glob(p, recursive=True)
        if res:
            return res[0]
    return None

cfg = find_first(["**/*.cfg", "performing_detection/**/*.cfg", "*/cfg/*.cfg"])
weights = find_first(["**/*.weights", "performing_detection/**/*.weights", "*.weights"])
names = find_first(["**/*.names", "**/*labels*.txt", "**/*.txt", "performing_detection/**/*.names", "performing_detection/**/*.txt"])

print("Auto-detect results (may be None):")
print(" CFG:", cfg)
print(" WEIGHTS:", weights)
print(" NAMES:", names)

# 4) If weights missing, attempt a repo Drive download (best-effort)
if weights is None:
    print("Weights not found locally — attempting to download a candidate from Drive (best-effort)...")
    try:
        import gdown
        DRIVE_ID = "1-Aam2D-fqnwecbeHwa4rtzxtNjwcDkP6"  # candidate id from repo README
        target = os.path.join("performing_detection","data","weights","crop_weed_detection.weights")
        os.makedirs(os.path.dirname(target), exist_ok=True)
        gdown.download(id=DRIVE_ID, output=target, quiet=False)
        if os.path.exists(target):
            weights = os.path.abspath(target)
            print("Downloaded weights to", weights)
    except Exception as e:
        print("Auto-download attempt failed:", e)

# 5) Final file resolution & defaults
cfg_path = os.path.abspath(cfg) if cfg else None
weights_path = os.path.abspath(weights) if weights else None
names_path = os.path.abspath(names) if names else None

print("\nResolved paths:")
print(" CFG:", cfg_path)
print(" WEIGHTS:", weights_path)
print(" NAMES:", names_path)

# 6) If critical files missing, warn but continue: Streamlit will allow uploading missing items
missing = []
if not cfg_path: missing.append("cfg (.cfg)")
if not weights_path: missing.append("weights (.weights)")
if not names_path: print("Label file not found; default labels will be used unless you upload one in the app.")

print("\nMissing (non-fatal unless weights/cfg are missing):", missing if missing else "none")

# 7) Write Streamlit app that can accept labels upload if missing
app_code = f'''
import streamlit as st, cv2, numpy as np, os
from PIL import Image

st.set_page_config(page_title="Crop & Weed Detection", layout="centered")
st.title("🌱 Crop & Weed Detection (YOLO + OpenCV)")

CFG_DEFAULT = r"{cfg_path}" if {str(bool(cfg_path))} else ""
WEIGHTS_DEFAULT = r"{weights_path}" if {str(bool(weights_path))} else ""
NAMES_DEFAULT = r"{names_path}" if {str(bool(names_path))} else ""

st.sidebar.header("Model files (detected)")
st.sidebar.write("CFG: " + (CFG_DEFAULT if CFG_DEFAULT else "Not found"))
st.sidebar.write("WEIGHTS: " + (WEIGHTS_DEFAULT if WEIGHTS_DEFAULT else "Not found"))
st.sidebar.write("NAMES: " + (NAMES_DEFAULT if NAMES_DEFAULT else "Not found"))
st.sidebar.write("---")

# Allow user to override/upload
cfg_upload = st.sidebar.file_uploader("Upload .cfg (optional)", type=["cfg"])
weights_upload = st.sidebar.file_uploader("Upload .weights (optional, large)", type=["weights"])
names_upload = st.sidebar.file_uploader("Upload labels (.names or .txt)", type=["names","txt"])

def save_uploaded(uploaded, target_name):
    if uploaded is None:
        return None
    out = os.path.join("/content", uploaded.name)
    with open(out, "wb") as f:
        f.write(uploaded.getbuffer())
    return out

if cfg_upload:
    CFG_PATH = save_uploaded(cfg_upload, cfg_upload.name)
elif CFG_DEFAULT:
    CFG_PATH = CFG_DEFAULT
else:
    CFG_PATH = None

if weights_upload:
    WEIGHTS_PATH = save_uploaded(weights_upload, weights_upload.name)
elif WEIGHTS_DEFAULT:
    WEIGHTS_PATH = WEIGHTS_DEFAULT
else:
    WEIGHTS_PATH = None

if names_upload:
    NAMES_PATH = save_uploaded(names_upload, names_upload.name)
elif NAMES_DEFAULT:
    NAMES_PATH = NAMES_DEFAULT
else:
    NAMES_PATH = None

# Load labels (fallback to default)
if NAMES_PATH and os.path.exists(NAMES_PATH):
    with open(NAMES_PATH, "r") as f:
        LABELS = [l.strip() for l in f.readlines() if l.strip()]
    st.sidebar.success("Loaded labels from " + os.path.basename(NAMES_PATH))
else:
    LABELS = ["crop","weed"]
    st.sidebar.warning("Labels file not found. Using default labels: ['crop','weed']. Upload a labels file to override.")

# Show main status
if not CFG_PATH or not WEIGHTS_PATH:
    st.error("Missing model CFG or weights. Upload them in the sidebar or place them in the repo folder.")
    st.stop()

# Load network
try:
    net = cv2.dnn.readNetFromDarknet(CFG_PATH, WEIGHTS_PATH)
    net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
    st.sidebar.success("YOLO model loaded")
except Exception as e:
    st.error(f"Failed to load YOLO model: {{e}}")
    st.stop()

# UI controls
conf_th = st.sidebar.slider("Confidence threshold", 0.1, 0.9, 0.4, 0.05)
nms_th = st.sidebar.slider("NMS threshold", 0.1, 0.6, 0.4, 0.05)
input_size = st.sidebar.selectbox("YOLO input size", [320,416,608], index=1)

# Upload / sample
uploaded = st.file_uploader("Upload image (jpg/png)", type=["jpg","jpeg","png"])
st.write("You can upload an image above. If you don't have labels file, upload it in the sidebar.")

if uploaded:
    image = Image.open(uploaded).convert("RGB")
    img = np.array(image)[:, :, ::-1].copy()  # RGB->BGR
    H,W = img.shape[:2]

    blob = cv2.dnn.blobFromImage(img, 1/255.0, (input_size,input_size), swapRB=True, crop=False)
    net.setInput(blob)
    ln = net.getLayerNames()
    try:
        out_names = [ln[i - 1] for i in net.getUnconnectedOutLayers().flatten()]
    except:
        out_names = [ln[i[0] - 1] for i in net.getUnconnectedOutLayers()]
    layerOutputs = net.forward(out_names)

    boxes, confs, classIDs = [], [], []
    for output in layerOutputs:
        for detection in output:
            scores = detection[5:]
            classID = int(np.argmax(scores))
            conf = float(scores[classID])
            if conf > conf_th:
                box = detection[0:4] * np.array([W,H,W,H])
                (centerX, centerY, bw, bh) = box.astype("int")
                x = int(centerX - bw/2); y = int(centerY - bh/2)
                boxes.append([x,y,int(bw),int(bh)]); confs.append(conf); classIDs.append(classID)

    idxs = cv2.dnn.NMSBoxes(boxes, confs, conf_th, nms_th)
    out_img = img.copy()
    if len(idxs)>0:
        for i in idxs.flatten():
            x,y,w,h = boxes[i]
            x1,y1,x2,y2 = max(0,x), max(0,y), min(W,x+w), min(H,y+h)
            label = LABELS[classIDs[i]] if classIDs[i] < len(LABELS) else f"class_{{classIDs[i]}}"
            color = tuple(map(int, np.random.randint(0,255,3).tolist()))
            cv2.rectangle(out_img, (x1,y1), (x2,y2), color, 2)
            cv2.putText(out_img, f"{{label}} {{confs[i]:.2f}}", (x1, max(15,y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    out_rgb = out_img[:, :, ::-1]
    st.image(out_rgb, use_column_width=True, caption="Detection result")
    st.success("Detection finished")
else:
    st.info("Upload an image to run detection.")
'''

open("app.py","w").write(app_code)
print("Wrote app.py")

# 8) Start Streamlit & ngrok
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()

# start streamlit in background and save logs
get_ipython().system_raw("streamlit run app.py --server.port 8501 > streamlit_logs.txt 2>&1 &")
time.sleep(2)
try:
    public_url = ngrok.connect(8501, "http")
    print("\n🌐 Streamlit public URL:", public_url)
    print("App is starting... if it does not load immediately wait 10-30 seconds.")
    print("To debug, run: !tail -n 200 streamlit_logs.txt")
except Exception as e:
    print("Failed to open ngrok tunnel:", e)
    print("Check your NGROK_AUTH_TOKEN and internet connectivity.")
    print("To inspect logs: !tail -n 200 streamlit_logs.txt")

# 9) Quick diagnostics show
print("\nDiagnostics:")
print("CFG exists:", os.path.exists(cfg_path) if 'cfg_path' in locals() else cfg is not None)
print("Weights exists:", os.path.exists(weights_path) if 'weights_path' in locals() else weights is not None)
print("Names exists:", os.path.exists(names_path) if 'names_path' in locals() else names is not None)
print("If any are False, upload missing files in the Streamlit sidebar or the Colab left Files panel.")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 108.7 MB/s eta 0:00:00
Working directory: /content/Crop_and_weed_detection
Auto-detect results (may be None):
 CFG: performing_detection/data/cfg/crop_weed.cfg
 WEIGHTS: performing_detection/data/weights/crop_weed_detection.weights
 NAMES: performing_detection/data/names/obj.names

Resolved paths:
 CFG: /content/Crop_and_weed_detection/performing_detection/data/cfg/crop_weed.cfg
 WEIGHTS: /content/Crop_and_weed_detection/performing_detection/data/weights/crop_weed_detection.weights
 NAMES: /content/Crop_and_weed_detection/performing_detection/data/names/obj.names

Missing (non-fatal unless weights/cfg are missing): none
Wrote app.py

🌐 Streamlit public URL: NgrokTunnel